# Phase 4 — LoRA+ Fine-Tuning — intent-classifier

This notebook trains and evaluates LoRA+ adapters for **2 models × 4 configs** on
Google Colab.

LoRA+ (Hayou et al., 2024) is identical to LoRA except that the B matrices are
trained at a **higher learning rate** than the A matrices
(`lr_B = loraplus_lr_ratio × lr_A`).  The adapter structure is unchanged, so
inference and evaluation code is exactly the same as LoRA.

Models: `qwen3-0.6b`, `llama3.2-1b` (both reach 98–100% with LoRA Config A–D).

**What runs here:**
1. Clone the repo from GitHub
2. Install dependencies
3. HF authentication (token stored in Colab Secrets)
4. GPU environment check
5. Configuration
6. Prepare data splits
7. Smoke test — 10 steps to validate the pipeline
8. Full training — all (model × config) combinations
9. Validation — load each adapter, evaluate on val split
10. Test evaluation (locked — run deliberately)
11. Inference sanity check
12. Download reports

**Note:** Plot generation is done offline. Reports are JSON files — download them,
then run `plot_loraplus_results.py` locally.

## 1. Clone repository

In [ ]:
import os

REPO_URL = "https://github.com/kon172verma/intent-classifier.git"
REPO_DIR = "/content/intent-classifier"

if not os.path.exists(REPO_DIR):
    os.system(f"git clone {REPO_URL} {REPO_DIR}")
else:
    os.system(f"git -C {REPO_DIR} pull")

print(f"Repo at: {REPO_DIR}")

## 2. Install dependencies

In [ ]:
%pip install -q \
    torch \
    "torchao>=0.16.0" \
    transformers \
    accelerate \
    "peft>=0.14.0" \
    trl \
    datasets \
    bitsandbytes \
    huggingface_hub \
    python-dotenv \
    sentencepiece \
    protobuf

print("Dependencies installed.")

## 3. Hugging Face authentication

Store your `HF_TOKEN` in **Colab Secrets** (key icon in the left sidebar).
Required for:
- Downloading `meta-llama/Llama-3.2-1B-Instruct` (gated model)
- Uploading trained adapters to `kon172verma/intent-classifier`

In [ ]:
import os

try:
    from google.colab import userdata  # type: ignore
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        print("HF_TOKEN loaded from Colab Secrets.")
    else:
        print("WARNING: HF_TOKEN secret is empty.")
except Exception as e:
    print(f"Not running in Colab or secret missing: {e}")

## 4. GPU environment check

In [ ]:
import subprocess
import platform
import torch

print(f"Python  : {platform.python_version()}")
print(f"PyTorch : {torch.__version__}")
print(f"CUDA    : {torch.version.cuda}")
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f"GPU     : {props.name}  ({props.total_memory / 1e9:.1f} GB VRAM)")
else:
    print("GPU     : not available — training will be very slow on CPU")

import peft
print(f"PEFT    : {peft.__version__}  (LoRA+ requires >= 0.9.0 for create_loraplus_optimizer)")

## 5. Configuration

Edit the variables below to control the experiment matrix.

| Variable | Description |
|---|---|
| `MODELS_TO_RUN` | Which models to train (subset of the 2 supported) |
| `CONFIGS_TO_RUN` | Which configs (A / B / C / D) |
| `DATASET_SIZE` | `"1k"` or `"10k"` |
| `DEVICE` | `"auto"` (recommended) or `"cuda"` |

**Config legend (same as LoRA — LoRA+ adds asymmetric LR per config):**
| Config | Description | loraplus_lr_ratio |
|---|---|---|
| A | Light — Q+V only, r=8 | 16 |
| B | Standard — full attention, r=16 | 16 |
| C | Wide — attn+MLP, r=16 | 16 |
| D | Heavy — attn+MLP, r=32 | 8 |

In [ ]:
import sys

REPO_DIR  = "/content/intent-classifier"
SRC_DIR   = f"{REPO_DIR}/finetune_LoRAplus/src"
DATA_DIR  = f"{REPO_DIR}/finetune_LoRAplus/data"

# ── Experiment matrix ──────────────────────────────────────────────────────
ALL_MODELS  = ["qwen3-0.6b", "llama3.2-1b"]
ALL_CONFIGS = ["A", "B", "C", "D"]

MODELS_TO_RUN  = ALL_MODELS   # change to a subset for partial runs
CONFIGS_TO_RUN = ALL_CONFIGS  # change to e.g. ["A", "B"] for partial

DATASET_SIZE = "1k"           # "1k" or "10k"
DEVICE       = "auto"

SPLIT_DIR = f"{DATA_DIR}/{DATASET_SIZE}"

# Add repo root to path so finetune_lib is importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

print("Configuration:")
print(f"  Models   : {MODELS_TO_RUN}")
print(f"  Configs  : {CONFIGS_TO_RUN}")
print(f"  Dataset  : {DATASET_SIZE}")
print(f"  Device   : {DEVICE}")

## 6. Prepare data splits

In [ ]:
import os
import subprocess
import sys

os.makedirs(SPLIT_DIR, exist_ok=True)

PREP_SCRIPT = f"{REPO_DIR}/finetune_LoRAplus/src/prepare_loraplus_data.py"
cmd = [
    sys.executable, "-u", PREP_SCRIPT,
    "--dataset-size", DATASET_SIZE,
    "--out-dir",      DATA_DIR,
]
print("Preparing data splits...")
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
if proc.returncode != 0:
    raise RuntimeError("prepare_loraplus_data.py failed")
print("Data preparation complete.")

## 7. Smoke test

Runs **10 training steps** on the first model + config A to validate the
complete pipeline (data loading → tokenisation → PEFT LoRA+ → training → report save)
without committing to a full run.

**Safe to skip** if you have already validated the pipeline.

In [ ]:
import subprocess
import sys
import time

SMOKE_MODEL  = MODELS_TO_RUN[0]
SMOKE_CONFIG = "A"

cmd = [
    sys.executable, "-u", f"{SRC_DIR}/loraplus_train.py",
    "--model",        SMOKE_MODEL,
    "--lora-config",  SMOKE_CONFIG,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
    "--smoke-test",
    "--no-push",
]
print("Running smoke test:", " ".join(cmd[2:]))
print()
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
elapsed = time.time() - t0
print(f"\nSmoke test {'PASSED' if proc.returncode == 0 else 'FAILED'} in {elapsed:.0f}s")

## 8. Train

Runs all (model × config) combinations via `run_loraplus_experiments.py`.

After each run the adapter is:
1. Saved locally → `finetune_LoRAplus/adapters/{run_tag}/`
2. Pushed to HF  → `kon172verma/intent-classifier/LoRA+/{run_tag}/`

Training reports (JSON with full `log_history`) are saved to
`finetune_LoRAplus/reports_training/`.

**Estimated runtimes on L4 GPU (1k dataset, similar to LoRA):**

| Model | Config A | Config B | Config C | Config D |
|---|---|---|---|---|
| qwen3-0.6b  | ~8 min  | ~9 min  | ~11 min | ~9 min  |
| llama3.2-1b | ~7 min  | ~8 min  | ~11 min | ~9 min  |

In [ ]:
import subprocess
import sys
import time

cmd = [
    sys.executable, "-u", f"{SRC_DIR}/run_loraplus_experiments.py",
    "--models",       *MODELS_TO_RUN,
    "--configs",      *CONFIGS_TO_RUN,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
]
print("Running:", " ".join(cmd[2:]))
print()
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
elapsed = time.time() - t0
print(f"\nAll training runs complete in {elapsed:.0f}s")
if proc.returncode != 0:
    print(f"WARNING: runner exited with code {proc.returncode}")

## 9. Validation

Evaluates each adapter on the validation split.
Reports are saved to `finetune_LoRAplus/reports_validation/`.

In [ ]:
import subprocess
import sys
import time

cmd = [
    sys.executable, "-u", f"{SRC_DIR}/run_loraplus_experiments.py",
    "--models",       *MODELS_TO_RUN,
    "--configs",      *CONFIGS_TO_RUN,
    "--dataset-size", DATASET_SIZE,
    "--device",       DEVICE,
    "--skip-training",  # adapters already trained above
]
print("Running validation:", " ".join(cmd[2:]))
print()
t0 = time.time()
proc = subprocess.Popen(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
    cwd=REPO_DIR,
)
_buf: list[str] = []
while True:
    ch = proc.stdout.read(1)
    if not ch:
        if _buf:
            line = "".join(_buf)
            if not line.startswith("Failed to load "):
                print(line, end="", flush=True)
        break
    _buf.append(ch)
    if ch in ("\r", "\n"):
        line = "".join(_buf)
        if not line.startswith("Failed to load "):
            print(line, end="", flush=True)
        _buf = []
proc.stdout.close()
proc.wait()
elapsed = time.time() - t0
print(f"\nAll validation runs complete in {elapsed:.0f}s")

## 10. Test evaluation (locked)

Run this cell deliberately **once** after reviewing validation results.

Reports saved to `finetune_LoRAplus/reports_test/`.

In [ ]:
import subprocess
import sys
import time

for model_key in MODELS_TO_RUN:
    for cfg in CONFIGS_TO_RUN:
        run_tag = f"{model_key}_{cfg}_{DATASET_SIZE}"
        cmd = [
            sys.executable, "-u", f"{SRC_DIR}/loraplus_validate.py",
            "--model",        model_key,
            "--lora-config",  cfg,
            "--dataset-size", DATASET_SIZE,
            "--split",        "test",
            "--device",       DEVICE,
        ]
        print(f"\nTest eval: {run_tag}")
        t0 = time.time()
        proc = subprocess.Popen(
            cmd,
            stdout=subprocess.PIPE,
            stderr=subprocess.STDOUT,
            text=True,
            bufsize=1,
            cwd=REPO_DIR,
        )
        _buf: list[str] = []
        while True:
            ch = proc.stdout.read(1)
            if not ch:
                if _buf:
                    line = "".join(_buf)
                    if not line.startswith("Failed to load "):
                        print(line, end="", flush=True)
                break
            _buf.append(ch)
            if ch in ("\r", "\n"):
                line = "".join(_buf)
                if not line.startswith("Failed to load "):
                    print(line, end="", flush=True)
                _buf = []
        proc.stdout.close()
        proc.wait()
        print(f"  done in {time.time()-t0:.0f}s  (exit={proc.returncode})")

## 11. Inference sanity check

Manually inspect model outputs on a few hand-picked examples.
This catches output format issues that aggregate accuracy metrics would not surface.

In [ ]:
import json
import sys
import torch
from pathlib import Path

# Pick any run_tag to inspect
INSPECT_MODEL  = MODELS_TO_RUN[0]
INSPECT_CONFIG = CONFIGS_TO_RUN[0]

if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

from finetune_lib import (
    FINETUNE_MODEL_REGISTRY, HF_HUB_REPO, hf_adapter_subfolder,
    build_chat_messages, apply_chat_template_safe,
    extract_prediction,
)
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

model_id  = FINETUNE_MODEL_REGISTRY[INSPECT_MODEL]
subfolder = hf_adapter_subfolder("LoRA+", INSPECT_MODEL, INSPECT_CONFIG, DATASET_SIZE)

print(f"Loading {INSPECT_MODEL} + LoRA+ adapter: {subfolder}")
tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=False)
base = AutoModelForCausalLM.from_pretrained(
    model_id, torch_dtype=torch.bfloat16, device_map="auto", trust_remote_code=False
)
model = PeftModel.from_pretrained(base, HF_HUB_REPO, subfolder=subfolder)
model.eval()

# Load a few val examples
val_path = Path(REPO_DIR) / "finetune_LoRAplus" / "data" / DATASET_SIZE / "val.jsonl"
examples = [json.loads(l) for l in val_path.read_text().splitlines()[:5]]

for ex in examples:
    messages = build_chat_messages(ex, INSPECT_MODEL)
    prompt   = apply_chat_template_safe(tokenizer, messages, INSPECT_MODEL)
    inputs   = tokenizer(prompt, return_tensors="pt").to(base.device)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=20, do_sample=False, pad_token_id=tokenizer.eos_token_id)
    decoded = tokenizer.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    pred    = extract_prediction(decoded)
    status  = "OK" if pred == ex["answer"] else "WRONG"
    print(f"  [{status}] answer={ex['answer']}  pred={pred}  raw={decoded!r}")

## 12. Download reports

Zips and downloads all JSON report files so you can run plot scripts locally.

Contents of the zip:
- `reports_training/` — training curves + log history
- `reports_validation/` — validation accuracy
- `reports_test/` — test accuracy + per-class metrics

In [ ]:
import os
import shutil
from pathlib import Path

try:
    from google.colab import files  # type: ignore

    lp_dir   = Path(REPO_DIR) / "finetune_LoRAplus"
    zip_base = "/content/loraplus_reports"

    report_dirs = {
        "reports_training":   lp_dir / "reports_training",
        "reports_validation": lp_dir / "reports_validation",
        "reports_test":       lp_dir / "reports_test",
    }

    # Create a staging folder and copy reports into it
    staging = Path("/content/loraplus_reports_staging")
    staging.mkdir(exist_ok=True)
    for name, src in report_dirs.items():
        dst = staging / name
        dst.mkdir(exist_ok=True)
        if src.exists():
            for f in src.glob("*.json"):
                shutil.copy(f, dst / f.name)

    zip_path = shutil.make_archive(zip_base, "zip", staging)
    print(f"Created {zip_path}")
    files.download(zip_path)
except ImportError:
    print("Not running in Colab — download the reports directory manually.")